# DeepFlow 합성데이터 B1/B2/B3 예측 실험 재현 노트북

이 노트북은 교수님께 실험 과정을 투명하게 보여주기 위한 Colab용 재현 파일입니다.

실행하는 내용:

1. 8개 기업별 합성데이터 CSV를 불러옵니다.
2. 각 기업-SKU의 마지막 6개월을 테스트 구간으로 분리합니다.
3. B1 3개월 평균 예측을 계산합니다.
4. B2 Pure LLM 예측 결과를 불러옵니다. 기본값은 입력 zip의 Gemini 캐시 결과이며, 설정을 바꾸면 Gemini API를 live 호출합니다.
5. B3 LGBM+도구 예측을 실행합니다.
6. 세 방식의 지표를 같은 기준으로 계산합니다.
7. 대표 5개 기업-SKU에 대해 실제값과 예측값을 시계열 그래프로 비교합니다.

주의:

- 이 실험은 실제 기업 데이터 검증이 아니라 합성데이터 기반 예비 실험입니다.
- B4 DeepFlow 전체 방식은 이 노트북에 포함하지 않았습니다.
- 수치 환각률과 실제 사용자 수용률은 별도 로그 설계가 필요하므로 여기서는 계산하지 않습니다.

## 0. Colab 실행 방법

1. Colab 링크를 엽니다.
2. 런타임 유형은 Python 3 그대로 둡니다.
3. 위에서 아래로 실행합니다.
4. 기본 실행은 Gemini API 키 없이 진행됩니다.

데이터 패키지는 공개 GitHub zip에서 자동으로 내려받기 때문에 별도 파일 업로드가 필요 없습니다.

기본값은 `RUN_LLM_CALLS = False`입니다. 이 경우 입력 zip에 포함된 캐시 B2 결과를 사용합니다.
교수님께 “현재 실행에서 LLM을 live 호출했다”고 말하려면 `RUN_LLM_CALLS = True`로 바꾸고 유효한 Gemini API 키를 입력해야 합니다.

In [ ]:
# 패키지 설치
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm", "scikit-learn", "pandas", "matplotlib", "seaborn", "requests"], check=False)
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False)

In [ ]:
# 기본 import 및 한글 폰트 설정
import json
import math
import os
import re
import time
import urllib.request
import zipfile
from getpass import getpass
from pathlib import Path

import lightgbm as lgb
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from IPython.display import display
from sklearn.metrics import mean_absolute_error

sns.set_theme(style="whitegrid")

def setup_korean_font():
    candidates = [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",
        "/Library/Fonts/NanumGothic.ttf",
    ]
    for font_path in candidates:
        if Path(font_path).exists():
            fm.fontManager.addfont(font_path)
            font_name = fm.FontProperties(fname=font_path).get_name()
            plt.rcParams["font.family"] = font_name
            plt.rcParams["axes.unicode_minus"] = False
            print(f"한글 폰트 설정 완료: {font_name}")
            return
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트 파일을 찾지 못했습니다. Colab에서는 fonts-nanum 설치 셀을 먼저 실행하세요.")

setup_korean_font()

TARGET = "synthetic_reference_demand"
OUTPUT_DIR = Path("/content/deepflow_colab_outputs") if IN_COLAB else Path("colab_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. 데이터 업로드 및 로딩

데이터 패키지는 공개 GitHub zip에서 자동으로 내려받습니다.

이 셀을 실행하면 8개 기업별 합성데이터 CSV와 기존 B2 캐시 결과가 자동으로 압축 해제됩니다.

In [ ]:
INPUT_PACKAGE_URL = "https://raw.githubusercontent.com/Jaeho777/Impactiveagent-demo/gh-pages/deepflow_colab_input_package.zip"

def download_input_package(data_root: Path) -> None:
    data_root.mkdir(parents=True, exist_ok=True)
    zip_path = data_root / "deepflow_colab_input_package.zip"
    if not zip_path.exists():
        print(f"입력 패키지 다운로드: {INPUT_PACKAGE_URL}")
        urllib.request.urlretrieve(INPUT_PACKAGE_URL, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_root)

# 데이터 패키지 자동 압축 해제 또는 로컬 경로 탐색
DATA_ROOT = Path("/content/deepflow_colab_data") if IN_COLAB else Path("colab/deepflow_colab_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

local_dataset_dir = Path("outputs/final_8_company_datasets_2026-05-23/datasets")
if local_dataset_dir.exists():
    DATASET_DIR = local_dataset_dir
    print(f"로컬 데이터셋 사용: {DATASET_DIR}")
else:
    if not list(DATA_ROOT.rglob("datasets/*.csv")):
        download_input_package(DATA_ROOT)

    candidates = [p for p in DATA_ROOT.rglob("datasets") if list(p.glob("*.csv"))]
    if not candidates:
        raise FileNotFoundError("datasets/*.csv를 찾지 못했습니다.")
    DATASET_DIR = candidates[0]
    print(f"다운로드 데이터셋 사용: {DATASET_DIR}")

csv_files = sorted(DATASET_DIR.glob("*.csv"))
print(f"CSV 파일 수: {len(csv_files)}")
for p in csv_files:
    print("-", p.name)

In [ ]:
# 데이터 로딩
def read_datasets(dataset_dir: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(dataset_dir.glob("*.csv")):
        frame = pd.read_csv(path)
        frame["dataset_file"] = path.name
        frames.append(frame)
    if not frames:
        raise RuntimeError(f"No CSV files found in {dataset_dir}")
    df = pd.concat(frames, ignore_index=True)
    df["month_dt"] = pd.to_datetime(df["month"] + "-01")
    df = df.sort_values(["company_id", "sku_family", "month_dt"]).reset_index(drop=True)
    return df

df_raw = read_datasets(DATASET_DIR)
print("전체 행 수:", len(df_raw))
print("기업 수:", df_raw["company_id"].nunique())
print("SKU 수:", df_raw[["company_id", "sku_family"]].drop_duplicates().shape[0])
print("월 범위:", df_raw["month"].min(), "~", df_raw["month"].max())
display(df_raw.head())

## 2. 예측 대상과 평가 구간

예측 대상은 `synthetic_reference_demand`입니다.

이 값은 정상 기준 출고량에 계절성, 추세, 외부 사건 영향, 잔차 변동을 반영한 최종 기준 출고량입니다.
보고서에서는 이 값을 **월간 출고량**으로 부릅니다.

평가 구간은 각 기업-SKU별 마지막 6개월입니다.

In [ ]:
# 시간 변수, lag 변수, train/validation/test 라벨 생성
CATEGORICAL_FEATURES = [
    "company_id",
    "industry_id",
    "industry_segment",
    "company_archetype",
    "sku_family",
    "event_type",
    "event_severity",
]

NUMERIC_FEATURES = [
    "base_monthly_demand",
    "event_demand_impact_pct",
    "event_lead_time_delta_days",
    "event_capacity_multiplier",
    "opening_inventory",
    "inbound_qty",
    "available_inventory",
    "effective_lead_time_days",
    "moq",
    "lot_multiple",
    "capacity_qty",
    "holding_cost_per_unit",
    "stockout_cost_per_unit",
    "lag_1",
    "lag_3",
    "lag_6",
    "rolling_mean_3",
    "rolling_mean_6",
    "month_sin",
    "month_cos",
    "time_index",
]

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["month_number"] = df["month_dt"].dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month_number"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month_number"] / 12)
    df["time_index"] = df.groupby(["company_id", "sku_family"]).cumcount()
    grouped = df.groupby(["company_id", "sku_family"], group_keys=False)
    df["lag_1"] = grouped[TARGET].shift(1)
    df["lag_3"] = grouped[TARGET].shift(3)
    df["lag_6"] = grouped[TARGET].shift(6)
    df["rolling_mean_3"] = grouped[TARGET].transform(lambda series: series.shift(1).rolling(3).mean())
    df["rolling_mean_6"] = grouped[TARGET].transform(lambda series: series.shift(1).rolling(6).mean())
    for col in ["lag_1", "lag_3", "lag_6", "rolling_mean_3", "rolling_mean_6"]:
        df[col] = df[col].fillna(df["base_monthly_demand"])
    return df

def assign_split(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["split"] = "train"
    for _, index in df.groupby(["company_id", "sku_family"]).groups.items():
        ordered = list(index)
        if len(ordered) < 18:
            raise RuntimeError("각 기업-SKU 시계열은 최소 18개월 이상이어야 합니다.")
        validation_index = ordered[-12:-6]
        test_index = ordered[-6:]
        df.loc[validation_index, "split"] = "validation"
        df.loc[test_index, "split"] = "test"
    return df

df = assign_split(add_time_features(df_raw))
split_summary = df.groupby("split").size().reset_index(name="rows")
display(split_summary)
print("테스트 행 수:", (df["split"] == "test").sum())

## 3. 공통 발주량 계산 및 지표 함수

B1과 B3는 같은 발주 계산식을 사용합니다.

B2는 LLM이 발주량을 직접 냅니다. 다만 제약 위반 여부와 비용은 같은 기준으로 사후 계산합니다.

In [ ]:
def ceil_to_multiple(value: float, multiple: float) -> int:
    multiple = max(1, int(round(multiple)))
    if value <= 0:
        return 0
    return int(math.ceil(value / multiple) * multiple)

def floor_to_multiple(value: float, multiple: float) -> int:
    multiple = max(1, int(round(multiple)))
    if value <= 0:
        return 0
    return int(math.floor(value / multiple) * multiple)

def constraint_violation(row: pd.Series, order_qty: int) -> tuple[bool, str]:
    moq = int(round(float(row["moq"])))
    lot = int(round(float(row["lot_multiple"])))
    capacity = int(round(float(row["capacity_qty"])))
    if order_qty <= 0:
        return False, ""
    if order_qty < moq:
        return True, "below_moq"
    if order_qty % lot != 0:
        return True, "lot_multiple"
    if order_qty > capacity:
        return True, "capacity"
    return False, ""

def cost_for_order(row: pd.Series, order_qty: int) -> tuple[float, int, int]:
    actual_demand = float(row[TARGET])
    available = float(row["available_inventory"])
    holding_cost = float(row["holding_cost_per_unit"])
    stockout_cost = float(row["stockout_cost_per_unit"])
    ending_inventory = max(0, available + order_qty - actual_demand)
    shortage_qty = max(0, actual_demand - available - order_qty)
    cost = ending_inventory * holding_cost + shortage_qty * stockout_cost
    return round(cost, 4), int(round(ending_inventory)), int(round(shortage_qty))

def recommend_order(row: pd.Series, forecast_demand: float) -> dict:
    lead_time = float(row["effective_lead_time_days"])
    moq = float(row["moq"])
    lot = float(row["lot_multiple"])
    capacity = float(row["capacity_qty"])
    available = float(row["available_inventory"])

    safety_stock = max(0, round(forecast_demand * (0.16 + min(lead_time, 35) / 170)))
    lead_time_demand = round(forecast_demand * lead_time / 30)
    target_position = max(safety_stock, lead_time_demand + safety_stock)
    raw_order = max(0, target_position - available)

    if raw_order <= 0:
        order_qty = 0
    else:
        order_qty = ceil_to_multiple(max(raw_order, moq), lot)
        order_qty = min(order_qty, floor_to_multiple(capacity, lot))

    recommended_cost, ending_inventory, shortage_qty = cost_for_order(row, order_qty)
    violation, violation_type = constraint_violation(row, order_qty)
    return {
        "recommended_order_qty": int(order_qty),
        "recommended_cost": recommended_cost,
        "projected_ending_inventory_model": ending_inventory,
        "projected_shortage_qty_model": shortage_qty,
        "constraint_violation": violation,
        "constraint_violation_type": violation_type,
    }

def order_error(reference: float, recommended: float) -> float:
    if reference == 0:
        return 0.0 if recommended == 0 else 1.0
    return abs(reference - recommended) / abs(reference)

def cost_gap(reference: float, recommended: float) -> float:
    if reference == 0:
        return 0.0 if recommended == 0 else 1.0
    return (recommended - reference) / abs(reference)

def build_method_rows(test_df: pd.DataFrame, method: str, forecasts: dict[int, float], orders: dict[int, int] | None = None, raw_source: dict[int, str] | None = None) -> pd.DataFrame:
    output = []
    for idx, row in test_df.iterrows():
        forecast = max(0, float(forecasts[idx]))
        if orders is None:
            rec = recommend_order(row, forecast)
            order_qty = int(rec["recommended_order_qty"])
            recommended_cost = float(rec["recommended_cost"])
            ending_inventory = int(rec["projected_ending_inventory_model"])
            shortage_qty = int(rec["projected_shortage_qty_model"])
            violation = bool(rec["constraint_violation"])
            violation_type = rec["constraint_violation_type"]
        else:
            order_qty = max(0, int(round(float(orders[idx]))))
            recommended_cost, ending_inventory, shortage_qty = cost_for_order(row, order_qty)
            violation, violation_type = constraint_violation(row, order_qty)

        actual = float(row[TARGET])
        reference_order = float(row["reference_order_qty"])
        reference_cost = float(row["reference_cost"])
        reference_decision = "order" if reference_order > 0 else "no_order"
        method_decision = "order" if order_qty > 0 else "no_order"

        output.append(
            {
                "row_index": idx,
                "method": method,
                "company_id": row["company_id"],
                "company_name": row["company_name"],
                "industry_id": row["industry_id"],
                "sku_family": row["sku_family"],
                "month": row["month"],
                "month_dt": row["month_dt"],
                "actual_demand": round(actual, 4),
                "forecast_demand": round(forecast, 4),
                "forecast_ape": round(abs(actual - forecast) / actual if actual else 0, 6),
                "reference_order_qty": int(round(reference_order)),
                "recommended_order_qty": order_qty,
                "order_error_pct": round(order_error(reference_order, float(order_qty)), 6),
                "constraint_violation": violation,
                "constraint_violation_type": violation_type,
                "reference_decision": reference_decision,
                "method_decision": method_decision,
                "decision_flip": reference_decision != method_decision,
                "reference_cost": round(reference_cost, 4),
                "recommended_cost": recommended_cost,
                "cost_gap_pct": round(cost_gap(reference_cost, float(recommended_cost)), 6),
                "projected_ending_inventory_model": ending_inventory,
                "projected_shortage_qty_model": shortage_qty,
                "available_inventory": int(round(float(row["available_inventory"]))),
                "effective_lead_time_days": int(round(float(row["effective_lead_time_days"]))),
                "moq": int(round(float(row["moq"]))),
                "lot_multiple": int(round(float(row["lot_multiple"]))),
                "capacity_qty": int(round(float(row["capacity_qty"]))),
                "event_type": row["event_type"],
                "event_demand_impact_pct": row["event_demand_impact_pct"],
                "raw_source": "" if raw_source is None else raw_source.get(idx, ""),
            }
        )
    return pd.DataFrame(output)

def summarize_metrics(combined: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for method, group in combined.groupby("method"):
        rows.append(
            {
                "method": method,
                "test_rows": len(group),
                "forecast_mape": round(float(group["forecast_ape"].mean()), 6),
                "forecast_mae": round(float(mean_absolute_error(group["actual_demand"], group["forecast_demand"])), 4),
                "order_error_pct": round(float(group["order_error_pct"].mean()), 6),
                "constraint_violation_rate": round(float(group["constraint_violation"].mean()), 6),
                "decision_flip_rate": round(float(group["decision_flip"].mean()), 6),
                "avg_recommended_cost": round(float(group["recommended_cost"].mean()), 4),
                "cost_gap_pct": round(float(group["cost_gap_pct"].mean()), 6),
            }
        )
    order = {"b1_three_month_avg": 1, "b2_pure_llm": 2, "b3_lgbm_tool": 3}
    metrics = pd.DataFrame(rows)
    metrics["sort_order"] = metrics["method"].map(order)
    return metrics.sort_values("sort_order").drop(columns=["sort_order"])

## 4. B1: 직전 3개월 평균 예측

B1은 가장 단순한 기준선입니다.

```text
B1 예측 출고량 = 직전 3개월 월간 출고량 평균
```

In [ ]:
def run_b1_three_month_average(df: pd.DataFrame) -> pd.DataFrame:
    test_df = df[df["split"] == "test"].copy()
    forecasts = {idx: float(row["rolling_mean_3"]) for idx, row in test_df.iterrows()}
    return build_method_rows(test_df, "b1_three_month_avg", forecasts)

b1_rows = run_b1_three_month_average(df)
print("B1 결과 행 수:", len(b1_rows))
display(b1_rows.head())

## 5. B3: LGBM 예측 + 발주 계산 도구

B3는 예측은 LGBM 모델이 수행하고, 추천 발주량은 공통 계산식으로 산출합니다.

이번 실행에서는 별도 검증 구간 튜닝을 하지 않습니다.
마지막 6개월 이전 데이터를 학습에 사용하고, 마지막 6개월을 테스트로 평가합니다.

In [ ]:
def run_b3_lgbm_tool(df: pd.DataFrame) -> tuple[pd.DataFrame, lgb.LGBMRegressor]:
    model_df = df.copy()
    train_df = model_df[model_df["split"].isin(["train", "validation"])].copy()
    test_df = model_df[model_df["split"] == "test"].copy()

    features = NUMERIC_FEATURES + CATEGORICAL_FEATURES
    x_train = pd.get_dummies(train_df[features], columns=CATEGORICAL_FEATURES)
    x_test = pd.get_dummies(test_df[features], columns=CATEGORICAL_FEATURES)
    x_test = x_test.reindex(columns=x_train.columns, fill_value=0)

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=240,
        learning_rate=0.045,
        num_leaves=31,
        min_child_samples=8,
        random_state=42,
        verbose=-1,
    )
    model.fit(x_train, train_df[TARGET])
    pred = model.predict(x_test)
    forecasts = dict(zip(test_df.index.tolist(), pred))
    return build_method_rows(test_df, "b3_lgbm_tool", forecasts), model

b3_rows, lgbm_model = run_b3_lgbm_tool(df)
print("B3 결과 행 수:", len(b3_rows))
display(b3_rows.head())

## 6. B2: Pure LLM 예측

B2는 LLM이 예측 출고량과 추천 발주량을 직접 내는 방식입니다.
기본 실행은 입력 zip에 포함된 Gemini 캐시 결과를 사용합니다.
`RUN_LLM_CALLS = True`로 바꾸면 Gemini API를 live 호출합니다.

중요한 구분:

- B2는 발주량을 계산식으로 보정하지 않습니다.
- LLM 출력 후 MOQ, 발주배수, 공급 가능량 위반 여부만 사후 검사합니다.
- API 키는 노트북 변수에만 입력하고 파일로 저장하지 않습니다.
- live 호출에서 403/429 등 API 문제가 생기면 기본 설정상 캐시 B2 결과로 자동 전환합니다.

In [ ]:
# LLM 실행 설정
# 기본값은 False입니다. 그래야 권한/쿼터 문제 없이 전체 실험 표와 그래프를 재현할 수 있습니다.
# True로 바꾸면 Gemini API를 live 호출합니다.
RUN_LLM_CALLS = False
USE_CACHED_B2_IF_RUN_LLM_FALSE = True
FALLBACK_TO_CACHED_B2_ON_API_ERROR = True
GEMINI_MODEL = "gemini-2.5-flash"

# RUN_LLM_CALLS=True면 Gemini API 키가 필요합니다.
# Colab에서 실행 시 입력창에 키를 넣으세요. 키는 출력/파일에 저장하지 않습니다.
if RUN_LLM_CALLS:
    if not os.environ.get("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = getpass("Gemini API Key 입력: ")

In [ ]:
def extract_json(text: str):
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        decoder = json.JSONDecoder()
        items = []
        position = 0
        while position < len(cleaned):
            match = re.search(r"[\[{]", cleaned[position:])
            if not match:
                break
            start = position + match.start()
            try:
                parsed, end = decoder.raw_decode(cleaned[start:])
            except json.JSONDecodeError:
                position = start + 1
                continue
            if isinstance(parsed, list):
                items.extend(parsed)
            else:
                items.append(parsed)
            position = start + end
        if items:
            return items
        match = re.search(r"(\[[\s\S]*\]|\{[\s\S]*\})", cleaned)
        if not match:
            raise
        return json.loads(match.group(1))

def flatten_items(parsed) -> list[dict]:
    if isinstance(parsed, list):
        return [item for item in parsed if isinstance(item, dict)]
    if isinstance(parsed, dict):
        for key in ["results", "items", "predictions", "forecasts", "data"]:
            if isinstance(parsed.get(key), list):
                return [item for item in parsed[key] if isinstance(item, dict)]
        return [parsed]
    return []

def prompt_for_b2_group(history: pd.DataFrame, test_rows: pd.DataFrame) -> str:
    history_items = [
        {
            "month": row.month,
            "monthly_shipment_qty": int(row.synthetic_reference_demand),
            "event_type": row.event_type,
            "event_demand_impact_pct": float(row.event_demand_impact_pct),
            "lead_time_days": int(row.effective_lead_time_days),
        }
        for row in history.sort_values("month_dt").tail(18).itertuples()
    ]

    target_rows = [
        {
            "target_no": position + 1,
            "row_index": int(index),
            "sku_family": row["sku_family"],
            "month": row["month"],
            "base_monthly_demand": int(row["base_monthly_demand"]),
            "event_type": row["event_type"],
            "event_severity": row["event_severity"],
            "event_demand_impact_pct": float(row["event_demand_impact_pct"]),
            "event_lead_time_delta_days": int(row["event_lead_time_delta_days"]),
            "event_capacity_multiplier": float(row["event_capacity_multiplier"]),
            "available_inventory": int(row["available_inventory"]),
            "effective_lead_time_days": int(row["effective_lead_time_days"]),
            "moq": int(row["moq"]),
            "lot_multiple": int(row["lot_multiple"]),
            "capacity_qty": int(row["capacity_qty"]),
        }
        for position, (index, row) in enumerate(test_rows.sort_values("month_dt").iterrows())
    ]

    meta = test_rows.iloc[0]
    payload = {
        "task": "For each target row, directly decide the forecast monthly shipment quantity and recommended order quantity. Do not call an external calculator. Return one strict JSON array only.",
        "company_id": meta["company_id"],
        "company_name": meta["company_name"],
        "industry_id": meta["industry_id"],
        "rules": [
            "forecast_demand must be an integer.",
            "recommended_order_qty must be an integer.",
            "Do not use thousands separators. Write 1500, not 1,500.",
            "recommended_order_qty may be 0 when no order is needed.",
            "Use MOQ, lot_multiple, capacity_qty as constraints, but do not explain.",
            "Return valid JSON only. Do not include markdown fences, comments, or prose.",
        ],
        "history_last_18_months": history_items,
        "target_rows": target_rows,
        "output_schema": [
            {
                "target_no": "integer",
                "row_index": "integer",
                "sku_family": "string",
                "month": "YYYY-MM",
                "forecast_demand": "integer",
                "recommended_order_qty": "integer",
            }
        ],
    }
    return json.dumps(payload, ensure_ascii=False)

def call_gemini_json(prompt: str, model: str = GEMINI_MODEL) -> str:
    api_key = os.environ.get("GEMINI_API_KEY", "")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY가 설정되지 않았습니다.")
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
    body = {
        "contents": [
            {
                "role": "user",
                "parts": [
                    {
                        "text": (
                            "You are a demand forecasting model. Return one strict JSON array only. "
                            "Do not include markdown fences or prose.\n\n"
                            + prompt
                        )
                    }
                ],
            }
        ],
        "generationConfig": {
            "temperature": 0,
            "responseMimeType": "application/json",
        },
    }
    response = requests.post(url, params={"key": api_key}, json=body, timeout=80)
    if response.status_code >= 400:
        raise RuntimeError(f"Gemini API error {response.status_code}: {response.text[:500]}")
    payload = response.json()
    return payload["candidates"][0]["content"]["parts"][0]["text"]

def merge_b2_items(parsed, test_rows: pd.DataFrame, forecasts: dict[int, float], orders: dict[int, int], raw_source: dict[int, str], raw_text: str) -> int:
    items = flatten_items(parsed)
    expected_indices = test_rows.index.tolist()
    expected_set = set(expected_indices)
    added = 0
    for position, item in enumerate(items):
        forecast = item.get("forecast_demand") or item.get("predicted_demand") or item.get("demand_forecast") or item.get("forecast")
        order = item.get("recommended_order_qty") or item.get("order_qty") or item.get("recommended_order")
        if forecast is None or order is None:
            continue

        resolved_index = None
        raw_row_index = item.get("row_index")
        if raw_row_index is not None:
            try:
                row_index = int(raw_row_index)
                if row_index in expected_set:
                    resolved_index = row_index
            except (TypeError, ValueError):
                pass
        if resolved_index is None and item.get("target_no") is not None:
            try:
                target_no = int(item["target_no"])
                if 1 <= target_no <= len(expected_indices):
                    resolved_index = int(expected_indices[target_no - 1])
            except (TypeError, ValueError):
                pass
        if resolved_index is None and len(items) == len(expected_indices) and position < len(expected_indices):
            resolved_index = int(expected_indices[position])
        if resolved_index is None:
            continue

        forecasts[resolved_index] = float(forecast)
        orders[resolved_index] = int(round(float(order)))
        raw_source[resolved_index] = raw_text
        added += 1
    return added

def find_cached_b2_file() -> Path | None:
    candidates = []
    for root in [DATA_ROOT, Path("."), Path("/content")]:
        if root.exists():
            candidates.extend(root.rglob("b2_pure_llm_predictions.csv"))
    return candidates[0] if candidates else None

B2_RESULT_SOURCE = "not_run"

def load_cached_b2_result(reason: str) -> pd.DataFrame:
    global B2_RESULT_SOURCE
    cached = find_cached_b2_file()
    if not cached:
        raise RuntimeError(f"캐시된 B2 결과를 찾지 못했습니다. 사유: {reason}")
    print(f"캐시된 B2 결과 사용: {cached}")
    print(f"사유: {reason}")
    cached_df = pd.read_csv(cached)
    if "month_dt" not in cached_df.columns:
        cached_df["month_dt"] = pd.to_datetime(cached_df["month"] + "-01")
    B2_RESULT_SOURCE = "cached_b2_from_input_zip"
    return cached_df

def run_b2_pure_llm(df: pd.DataFrame) -> pd.DataFrame:
    global B2_RESULT_SOURCE
    test_df = df[df["split"] == "test"].copy()
    raw_dir = OUTPUT_DIR / "b2_raw_llm_responses"
    raw_dir.mkdir(parents=True, exist_ok=True)

    if not RUN_LLM_CALLS:
        if USE_CACHED_B2_IF_RUN_LLM_FALSE:
            return load_cached_b2_result("RUN_LLM_CALLS=False")
        raise RuntimeError("RUN_LLM_CALLS=False인데 캐시된 B2 결과를 찾지 못했습니다.")

    forecasts: dict[int, float] = {}
    orders: dict[int, int] = {}
    raw_source: dict[int, str] = {}
    logs = []

    groups = list(df.groupby(["company_id", "sku_family"]))
    for number, ((company_id, sku_family), group) in enumerate(groups, start=1):
        group = group.sort_values("month_dt")
        test_rows = group[group["split"] == "test"].copy()
        history = group[group["split"].isin(["train", "validation"])].copy()
        prompt = prompt_for_b2_group(history, test_rows)
        try:
            raw_text = call_gemini_json(prompt)
            (raw_dir / f"{company_id}_{sku_family}.json.txt").write_text(raw_text, encoding="utf-8")
            parsed = extract_json(raw_text)
            added = merge_b2_items(parsed, test_rows, forecasts, orders, raw_source, raw_text)
            missing = [idx for idx in test_rows.index.tolist() if idx not in forecasts or idx not in orders]
            status = "ok" if not missing else "partial"
            message = f"added={added}, missing={len(missing)}"
        except Exception as error:
            status = "fail"
            message = str(error)[:500]
        logs.append({"company_id": company_id, "sku_family": sku_family, "status": status, "message": message})
        print(f"[{number}/{len(groups)}] {company_id} {sku_family}: {status} {message}")
        pd.DataFrame(logs).to_csv(OUTPUT_DIR / "b2_pure_llm_call_log.csv", index=False, encoding="utf-8-sig")
        time.sleep(0.25)

    missing = [idx for idx in test_df.index.tolist() if idx not in forecasts or idx not in orders]
    if missing:
        display(pd.DataFrame(logs))
        if FALLBACK_TO_CACHED_B2_ON_API_ERROR:
            return load_cached_b2_result(f"Gemini live 호출 결과가 불완전합니다. missing_rows={len(missing)}")
        raise RuntimeError(f"B2 Pure LLM missing forecasts/orders for {len(missing)} rows.")

    rows = build_method_rows(test_df, "b2_pure_llm", forecasts, orders, raw_source)
    B2_RESULT_SOURCE = "live_gemini_api"
    return rows

In [ ]:
b2_rows = run_b2_pure_llm(df)
print("B2 결과 행 수:", len(b2_rows))
print("B2 결과 출처:", B2_RESULT_SOURCE)
display(b2_rows.head())

## 7. 방식별 결과 통합 및 지표 계산

In [ ]:
combined = pd.concat([b1_rows, b2_rows, b3_rows], ignore_index=True)
metrics = summarize_metrics(combined)

# 결과 저장
b1_rows.to_csv(OUTPUT_DIR / "b1_three_month_avg_predictions.csv", index=False, encoding="utf-8-sig")
b2_rows.to_csv(OUTPUT_DIR / "b2_pure_llm_predictions.csv", index=False, encoding="utf-8-sig")
b3_rows.to_csv(OUTPUT_DIR / "b3_lgbm_tool_predictions.csv", index=False, encoding="utf-8-sig")
combined.to_csv(OUTPUT_DIR / "combined_b1_b2_b3_predictions.csv", index=False, encoding="utf-8-sig")
metrics.to_csv(OUTPUT_DIR / "metrics_b1_b2_b3.csv", index=False, encoding="utf-8-sig")

display(metrics)

print("결과 저장 위치:", OUTPUT_DIR)

## 8. 검산

아래 검산에서 결측이 있으면 결과를 발표하면 안 됩니다.

In [ ]:
check_rows = []
for name, frame in [
    ("B1", b1_rows),
    ("B2", b2_rows),
    ("B3", b3_rows),
    ("combined", combined),
]:
    check_rows.append(
        {
            "name": name,
            "rows": len(frame),
            "forecast_missing": int(frame["forecast_demand"].isna().sum()),
            "order_missing": int(frame["recommended_order_qty"].isna().sum()),
            "constraint_violations": int(frame["constraint_violation"].sum()),
        }
    )
check_df = pd.DataFrame(check_rows)
display(check_df)

assert len(b1_rows) == 144
assert len(b2_rows) == 144
assert len(b3_rows) == 144
assert len(combined) == 432
assert check_df[["forecast_missing", "order_missing"]].to_numpy().sum() == 0
print("검산 통과")

## 9. 대표 5개 시계열 예측 비교 시각화

각 그래프는 한 기업-SKU의 실제 월간 출고량과 B1/B2/B3 예측값을 비교합니다.

- 회색 실선: 과거 및 테스트 구간 실제 월간 출고량
- 점선: 각 방식의 테스트 6개월 예측값

In [ ]:
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def select_representative_keys(combined: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    # 예측 난이도가 보이는 사례를 고르기 위해 B1 MAPE가 큰 기업-SKU를 우선 선택합니다.
    b1 = combined[combined["method"] == "b1_three_month_avg"].copy()
    rank = (
        b1.groupby(["company_id", "company_name", "sku_family"], as_index=False)
        .agg(mean_b1_mape=("forecast_ape", "mean"), mean_actual=("actual_demand", "mean"))
        .sort_values(["mean_b1_mape", "mean_actual"], ascending=[False, False])
        .head(top_n)
    )
    return rank

selected = select_representative_keys(combined, 5)
display(selected)

method_labels = {
    "b1_three_month_avg": "B1 3개월 평균",
    "b2_pure_llm": "B2 Pure LLM",
    "b3_lgbm_tool": "B3 LGBM+도구",
}
method_colors = {
    "b1_three_month_avg": "#6b7280",
    "b2_pure_llm": "#dc2626",
    "b3_lgbm_tool": "#2563eb",
}

figure_paths = []
for i, row in enumerate(selected.itertuples(index=False), start=1):
    company_id = row.company_id
    sku_family = row.sku_family
    company_name = row.company_name

    actual_series = df[(df["company_id"] == company_id) & (df["sku_family"] == sku_family)].copy()
    actual_series = actual_series.sort_values("month_dt").tail(24)
    pred_series = combined[(combined["company_id"] == company_id) & (combined["sku_family"] == sku_family)].copy()
    pred_series["month_dt"] = pd.to_datetime(pred_series["month"] + "-01")

    plt.figure(figsize=(13, 5.5))
    plt.plot(
        actual_series["month_dt"],
        actual_series[TARGET],
        color="#111827",
        linewidth=2.4,
        marker="o",
        label="실제 월간 출고량",
    )

    test_start = pred_series["month_dt"].min()
    plt.axvspan(test_start, pred_series["month_dt"].max(), color="#fef3c7", alpha=0.35, label="테스트 구간")

    for method, label in method_labels.items():
        one = pred_series[pred_series["method"] == method].sort_values("month_dt")
        plt.plot(
            one["month_dt"],
            one["forecast_demand"],
            linestyle="--",
            linewidth=2,
            marker="s",
            color=method_colors[method],
            label=label,
        )

    plt.title(f"시계열 예측 비교 {i}: {company_name} / {sku_family}", fontsize=15, pad=14)
    plt.xlabel("월")
    plt.ylabel("월간 출고량")
    plt.legend(loc="best")
    plt.tight_layout()
    path = FIG_DIR / f"timeseries_compare_{i}_{company_id}_{sku_family}.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_paths.append(path)

print("저장된 시각화 파일")
for path in figure_paths:
    print("-", path)

## 10. 교수님께 설명할 때 사용할 문장

아래 문장만 사용하면 과장 없이 설명할 수 있습니다.

In [ ]:
briefing = f'''
합성데이터 8개 기업, 24개 SKU에 대해 각 기업-SKU의 마지막 6개월을 테스트 구간으로 두었습니다.
B1은 직전 3개월 평균, B2는 Gemini LLM 직접 예측 결과, B3는 LGBM 예측 후 발주 계산식 적용 방식입니다.
현재 노트북의 B2 결과 출처는 {B2_RESULT_SOURCE}입니다.
세 방식 모두 방식별 {len(b1_rows)}개 테스트 결과를 산출했고, 예측 MAPE, 발주량 오차, 제약 위반율, 의사결정 flip, 비용 지표를 같은 기준으로 계산했습니다.
현재 결과는 실제 기업 데이터 성능 검증이 아니라 합성데이터 기반 예비 실험입니다.
다만 논문 5번의 벤치마크 결과표를 채우기 위한 실험 절차는 입력 데이터부터 예측, 발주량 산출, 제약 검증, 지표 계산까지 한 번 연결해서 실행했습니다.
'''
print("\n".join(line.strip() for line in briefing.strip().splitlines()))

## 11. 결과 파일 압축

Colab 실행 후 아래 셀을 실행하면 결과 CSV와 그래프 PNG를 한 번에 내려받을 수 있습니다.

In [ ]:
zip_output = OUTPUT_DIR.parent / "deepflow_benchmark_colab_outputs.zip"
with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(OUTPUT_DIR.parent))
print("압축 파일:", zip_output)

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_output))